In [1]:
!git clone https://github.com/YutingLi0606/Idempotent-Continual-Learning

Cloning into 'Idempotent-Continual-Learning'...
remote: Enumerating objects: 2149, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 2149 (delta 87), reused 31 (delta 31), pack-reused 2026 (from 7)
Receiving objects: 100% (2149/2149), 1.22 GiB | 44.45 MiB/s, done.
Resolving deltas: 100% (331/331), done.
Updating files: 100% (3100/3100), done.


In [4]:
import os
os.chdir('/kaggle/working/Idempotent-Continual-Learning')

In [2]:
!pip install mlflow setproctitle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 20.5 MB/s eta 0:00:00


In [5]:
code = """import torch
import copy
from models.utils.continual_model import ContinualModel
from utils.args import add_management_args, add_experiment_args, add_rehearsal_args, ArgumentParser
from utils.buffer import Buffer
import torch.nn.functional as F
from utils.args import *

def add_parser(parser):
    parser.add_argument('--weighta', type=float, help='Penalty weight.')
    parser.add_argument('--weightb', type=float, help='Penalty weight.')
    parser.add_argument('--weightc', type=float, help='Penalty weight.')
    parser.add_argument('--weightmask', type=float, help='Penalty weight.')
    parser.add_argument('--adaptive_weight', type=str2bool, default=False)
    parser.add_argument('--adaptive_gamma', type=float, default=0.5)
    parser.add_argument('--uncertainty_sampling', type=str2bool, default=False)
    parser.add_argument('--class_balance', type=str2bool, default=True)
    return parser

def get_parser():
    parser = ArgumentParser(description='IDER')
    add_management_args(parser)
    add_experiment_args(parser)
    add_rehearsal_args(parser)
    parser = add_parser(parser)
    return parser

class Ider(ContinualModel):
    NAME = 'ider'
    COMPATIBILITY = ['class-il', 'domain-il', 'task-il', 'general-continual']

    def __init__(self, backbone, loss, args, transform):
        super(Ider, self).__init__(backbone, loss, args, transform)
        self.buffer = Buffer(self.args.buffer_size, self.device, class_balance=self.args.class_balance)
        self.ft = True
        self.task = 0
        self.num_classs = backbone.num_classes
        self.s = backbone.num_classes
        self.first_task = True
        self.old_model = self.deepcopy_model(self.net)

    def get_adaptive_weight(self, base_weight, n_tasks):
        if self.args.adaptive_weight and self.task > 0:
            scale = 1.0 + self.args.adaptive_gamma * (self.task / n_tasks)
            return base_weight * scale
        return base_weight

    def get_uncertain_samples(self, buf_inputs, buf_labels, num_samples):
        with torch.no_grad():
            batch_size = buf_inputs.shape[0]
            y_0 = torch.ones(batch_size, self.num_classs).to(self.device) / self.s
            z = self.net.f1(buf_inputs)
            y_1, _ = self.net.f2(z, y_0)
            z_old = self.old_model.f1(buf_inputs)
            y_2, _ = self.old_model.f2(z_old, y_1.softmax(-1))
            # idempotence distance = uncertainty score
            scores = torch.norm(y_1 - y_2, dim=1)
            # sample proportional to uncertainty
            probs = (scores / scores.sum()).cpu()
            indices = torch.multinomial(probs, num_samples, replacement=False)
        return buf_inputs[indices], buf_labels[indices]

    def observe(self, inputs, labels, not_aug_inputs):
        batch_size, _, H, W = inputs.shape
        n_tasks = self.args.n_tasks
        weighta = self.get_adaptive_weight(self.args.weighta, n_tasks)
        weightb = self.get_adaptive_weight(self.args.weightb, n_tasks)

        self.opt.zero_grad()
        mask_current = torch.rand(1) > self.args.weightmask
        y_0_current = F.one_hot(labels, self.num_classs).float() if mask_current else torch.ones(batch_size, self.num_classs).to(self.device) / self.s
        z_current = self.net.f1(inputs)
        y_1_current, z1_current = self.net.f2(z_current, y_0_current)
        y_2_current, z2_current = self.net.f2(z_current, y_1_current.softmax(-1))
        loss = 0.5 * (self.loss(y_1_current, labels) + self.loss(y_2_current, labels))

        if weightb != 0 and self.task > 0:
            y_current_mask = torch.ones(batch_size, self.num_classs).to(self.device) / self.s
            z = self.net.f1(inputs)
            y_1, z1 = self.net.f2(z, y_current_mask)
            z_old = self.old_model.f1(inputs)
            y_2, z2 = self.old_model.f2(z_old, y_1.softmax(-1))
            loss += weightb * F.mse_loss(y_1, y_2)

        if not self.buffer.is_empty() and self.args.weightc != 0:
            buf_inputs, buf_labels, _, _, _ = self.buffer.get_data(self.args.minibatch_size, transform=self.transform)
            batch_size, _, H, W = buf_inputs.shape
            mask = torch.rand(1) > self.args.weightmask
            y_0_buf = F.one_hot(buf_labels, self.num_classs).float() if mask else torch.ones(batch_size, self.num_classs).to(self.device) / self.s
            z_buf = self.net.f1(buf_inputs)
            y_1_buf, _ = self.net.f2(z_buf, y_0_buf)
            y_2_buf, _ = self.net.f2(z_buf, y_1_buf.softmax(-1))
            loss += self.args.weightc * (self.loss(y_1_buf, buf_labels) + self.loss(y_2_buf, buf_labels))

        if not self.buffer.is_empty() and self.task > 0 and weighta != 0:
            buf_inputs, buf_labels, _, _, _ = self.buffer.get_data(self.args.minibatch_size, transform=self.transform)
            batch_size, _, H, W = buf_inputs.shape

            # uncertainty sampling — prioritize forgotten samples
            if self.args.uncertainty_sampling and buf_inputs.shape[0] > self.args.minibatch_size:
                buf_inputs, buf_labels = self.get_uncertain_samples(buf_inputs, buf_labels, self.args.minibatch_size)
                batch_size = buf_inputs.shape[0]

            y_0 = torch.ones(batch_size, self.num_classs).to(self.device) / self.s
            z = self.net.f1(buf_inputs)
            y_1, _ = self.net.f2(z, y_0)
            z_old = self.old_model.f1(buf_inputs)
            y_2, _ = self.old_model.f2(z_old, y_1.softmax(-1))
            loss += weighta * F.mse_loss(y_1, y_2)

        loss.backward()
        self.opt.step()
        self.buffer.add_data(examples=not_aug_inputs, labels=labels, logits=y_1_current.data, logits2=y_2_current.data, mask=y_0_current)
        return loss.item()

    def end_task(self, dataset):
        print('\\n\\n')
        self.task += 1
        print(self.task)
        if self.first_task:
            self.first_task = False
            self.old_model = self.deepcopy_model(self.net).to(self.device)
        else:
            self.old_model = self.deepcopy_model(self.net).to(self.device)

    @staticmethod
    def deepcopy_model(model):
        return copy.deepcopy(model)
"""

with open('models/ider.py', 'w') as f:
    f.write(code)
print("Done!")

Done!


In [6]:
!head -20 models/ider.py

import torch
import copy
from models.utils.continual_model import ContinualModel
from utils.args import add_management_args, add_experiment_args, add_rehearsal_args, ArgumentParser
from utils.buffer import Buffer
import torch.nn.functional as F
from utils.args import *

def add_parser(parser):
    parser.add_argument('--weighta', type=float, help='Penalty weight.')
    parser.add_argument('--weightb', type=float, help='Penalty weight.')
    parser.add_argument('--weightc', type=float, help='Penalty weight.')
    parser.add_argument('--weightmask', type=float, help='Penalty weight.')
    parser.add_argument('--adaptive_weight', type=str2bool, default=False)
    parser.add_argument('--adaptive_gamma', type=float, default=0.5)
    parser.add_argument('--uncertainty_sampling', type=str2bool, default=False)
    parser.add_argument('--class_balance', type=str2bool, default=True)
    return parser

def get_parser():


In [7]:
sh_content = """for N_TASKS in 5
do
    for SEED in 0
    do
    python main.py --model="ider" --load_best_args --class_balance=True --dataset="seq-cifar10" --seed=$SEED --n_tasks=$N_TASKS --buffer_size=200 --n_epochs=30 --adaptive_weight=True --adaptive_gamma=0.5 --uncertainty_sampling=True --run_name="ider+adaptive+uncertainty gamma0.5 seed $SEED" --experiment_name="cifar10/ider_improved_g0.5"
    python main.py --model="ider" --load_best_args --class_balance=True --dataset="seq-cifar10" --seed=$SEED --n_tasks=$N_TASKS --buffer_size=200 --n_epochs=30 --adaptive_weight=True --adaptive_gamma=1.0 --uncertainty_sampling=True --run_name="ider+adaptive+uncertainty gamma1.0 seed $SEED" --experiment_name="cifar10/ider_improved_g1.0"
    done
done
"""

with open('run_cifar10_improved.sh', 'w') as f:
    f.write(sh_content)
print("Done!")

Done!


In [8]:
!cat run_cifar10_improved.sh

for N_TASKS in 5
do
    for SEED in 0
    do
    python main.py --model="ider" --load_best_args --class_balance=True --dataset="seq-cifar10" --seed=$SEED --n_tasks=$N_TASKS --buffer_size=200 --n_epochs=30 --adaptive_weight=True --adaptive_gamma=0.5 --uncertainty_sampling=True --run_name="ider+adaptive+uncertainty gamma0.5 seed $SEED" --experiment_name="cifar10/ider_improved_g0.5"
    python main.py --model="ider" --load_best_args --class_balance=True --dataset="seq-cifar10" --seed=$SEED --n_tasks=$N_TASKS --buffer_size=200 --n_epochs=30 --adaptive_weight=True --adaptive_gamma=1.0 --uncertainty_sampling=True --run_name="ider+adaptive+uncertainty gamma1.0 seed $SEED" --experiment_name="cifar10/ider_improved_g1.0"
    done
done


In [ ]:
!bash run_cifar10_improved.sh

ider
50
use changed model middle
backbone number of parameters = 11848906
Using class balanced buffer
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
100%|████████████████████████████████████████| 170M/170M [00:01<00:00, 91.8MB/s]
evaluation acc:
[0.0, 0.0, 0.0, 50.0, 0.0]

[ 03-27 | 09:22 ] Task 0 | epoch 49: |██████████████████████████████████████████████████| 110.24 ep/h | loss: 0.00492246 |


1
evaluation acc:
[98.9]
evaluation acc: [98.9]
ECE per loader: [0.7642795797437429]
Mean ECE: 0.7643

Accuracy for 1 task(s): 	 [Class-IL]: 98.9 % 	 [Task-IL]: 98.9 %

evaluation acc:
[0.0]
[ 03-27